In [1]:
import pandas as pd
import numpy as np

# 1. Load the two datasets
print("Loading data...")
df_general = pd.read_csv('Hospital_General_Information.csv')
df_visits = pd.read_csv('Unplanned_Hospital_Visits-Hospital.csv')

# 2. Filter for the "Hospital-Wide All-Cause Readmission" rate
# CMS uses the Measure ID 'Hybrid_HWR' for overall 30-day readmissions
df_readmission = df_visits[df_visits['Measure ID'] == 'Hybrid_HWR'].copy()

# 3. Clean the Readmission Score (convert from text to numbers)
# CMS puts "Not Available" for some hospitals, we need to handle that
df_readmission['Score'] = pd.to_numeric(df_readmission['Score'], errors='coerce')

# 4. Merge the General Info with the Readmission Data
# We join them using the unique 'Facility ID'
df_merged = pd.merge(
    df_general[['Facility ID', 'Facility Name', 'State', 'City/Town', 'Hospital overall rating']],
    df_readmission[['Facility ID', 'Score']],
    on='Facility ID',
    how='inner'
)

# Rename columns to be dashboard-friendly
df_merged.rename(columns={
    'Hospital overall rating': 'Overall Rating',
    'Score': 'Readmission Rate (%)'
}, inplace=True)

# 5. Create a Mock "Estimated Penalty" calculation for the demo
# In reality, this requires complex Medicare claims data, but for our dashboard, 
# we will estimate it based on how far above the national average (approx 15.0%) they are.
national_avg = 15.0
df_merged['Penalty Flag'] = df_merged['Readmission Rate (%)'] > national_avg

# If they are above average, estimate a penalty between $100k - $800k based on severity
np.random.seed(42) # Keeping it consistent for the dashboard
df_merged['Estimated Penalty ($)'] = np.where(
    df_merged['Penalty Flag'],
    (df_merged['Readmission Rate (%)'] - national_avg) * 150000 + np.random.randint(50000, 200000, len(df_merged)),
    0
)

# Clean up data types and drop NaN readmission rates
df_merged = df_merged.dropna(subset=['Readmission Rate (%)'])
df_merged['Estimated Penalty ($)'] = df_merged['Estimated Penalty ($)'].round(0)

# 6. Let's test it out with some Texas Hospitals!
print("\n--- Data Cleaning Complete ---")
texas_hospitals = df_merged[df_merged['State'] == 'TX'].sort_values('Readmission Rate (%)', ascending=False)
texas_hospitals.head(10)

Loading data...

--- Data Cleaning Complete ---


/var/folders/pz/qq_g0z9j55vd4plml29lz6n40000gn/T/ipykernel_15812/290872207.py:7: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df_visits = pd.read_csv('Unplanned_Hospital_Visits-Hospital.csv')


,Facility ID,Facility Name,State,City/Town,Overall Rating,Readmission Rate (%),Penalty Flag,Estimated Penalty ($)
4080,450672,MEDICAL CITY FORT WORTH,TX,FORT WORTH,2,16.9,True,343597.0
4002,450222,HCA HOUSTON HEALTHCARE CONROE,TX,CONROE,2,16.9,True,379825.0
4036,450447,NAVARRO REGIONAL HOSPITAL,TX,CORSICANA,2,16.6,True,376694.0
4068,450651,MEDICAL CITY PLANO,TX,PLANO,2,16.5,True,321151.0
4100,450730,CARROLLTON REGIONAL MEDICAL CENTER,TX,CARROLLTON,2,16.5,True,286669.0
4043,450484,WOODLAND HEIGHTS MEDICAL CENTER,TX,LUFKIN,3,16.4,True,325982.0
3944,450032,CHRISTUS GOOD SHEPHERD MEDICAL CENTER,TX,LONGVIEW,3,16.4,True,394327.0
4084,450678,WHITE ROCK MEDICAL CENTER,TX,DALLAS,2,16.4,True,399335.0
4737,670080,SETON MEDICAL CENTER HARKER HEIGHTS,TX,HARKER HEIGHTS,4,16.3,True,269704.0
4060,450617,HCA HOUSTON HEALTHCARE CLEAR LAKE,TX,WEBSTER,3,16.2,True,374972.0
